In [1]:
import pandas as pd
import numpy as np

In [4]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\Air_Quality_Data\AQI_daily_2024_Pusa_Delhi_DPCC_2024.xlsx")

In [5]:
df

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,375.0,168.0,198.0,136.0,226.0,239.0,116.0,56.0,125.0,151.0,375.0,259.0
1,2,357.0,207.0,NaN,152.0,208.0,198.0,147.0,56.0,70.0,150.0,286.0,283.0
2,3,358.0,285.0,132.0,145.0,299.0,165.0,111.0,46.0,58.0,142.0,364.0,237.0
3,4,395.0,324.0,153.0,159.0,334.0,218.0,53.0,40.0,70.0,148.0,360.0,189.0
4,5,360.0,210.0,142.0,168.0,254.0,249.0,69.0,33.0,83.0,135.0,354.0,169.0
5,6,329.0,194.0,141.0,185.0,283.0,185.0,80.0,46.0,80.0,121.0,326.0,183.0
6,7,343.0,275.0,180.0,192.0,266.0,237.0,108.0,53.0,43.0,135.0,360.0,214.0
7,8,360.0,199.0,177.0,166.0,206.0,257.0,86.0,34.0,103.0,159.0,350.0,227.0
8,9,368.0,205.0,155.0,150.0,152.0,168.0,82.0,49.0,147.0,159.0,337.0,196.0
9,10,310.0,282.0,190.0,259.0,148.0,164.0,123.0,48.0,104.0,148.0,329.0,265.0


In [6]:
df_cleaned = df.drop_duplicates(keep='first')
df_cleaned.shape

(41, 13)

In [7]:
# Replace 'NA', empty strings, and explicit NaNs with np.nan for consistency
df_cleaned = df_cleaned.replace('NA', np.nan)
df_cleaned = df_cleaned.replace(r'^\s*$', np.nan, regex=True)

# Convert all columns except 'Day' to numeric
for col in df_cleaned.columns:
    if col != 'Day':
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')

# Fill missing values with the mean of each column
df_filled = df_cleaned.fillna(df_cleaned.mean(numeric_only=True))

In [8]:
def handle_outliers_iqr(df):
    for col in df.columns:
        if col != 'Day' and df[col].dtype != 'O':
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            mean_val = df[col].mean()
            df[col] = np.where((df[col] < lower) | (df[col] > upper), mean_val, df[col])
    return df

df_no_outliers = handle_outliers_iqr(df_filled.copy())

In [9]:
# Drop non-feature rows if present, and reset index
df_ml_ready = df_no_outliers.copy()

# If your dataset contains summary/statistical rows (not data for 'Day'), remove them
df_ml_ready = df_ml_ready[df_ml_ready['Day'].apply(lambda x: str(x).isdigit())]
df_ml_ready = df_ml_ready.reset_index(drop=True)

# Optional: Convert 'Day' to int if needed
df_ml_ready['Day'] = df_ml_ready['Day'].astype(int)

df_ml_ready

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,375.000000,168.000000,198.000,136.00000,226.000000,239.000000,116.000000,56.000000,125.000000,151.000000,375.000000,259.000000
1,2,357.000000,207.000000,152.625,152.00000,208.000000,198.000000,147.000000,56.000000,70.000000,150.000000,286.000000,283.000000
2,3,358.000000,285.000000,132.000,145.00000,299.000000,165.000000,111.000000,46.000000,58.000000,142.000000,364.000000,237.000000
3,4,395.000000,324.000000,153.000,159.00000,194.485714,218.000000,53.000000,40.000000,70.000000,148.000000,360.000000,189.000000
4,5,360.000000,210.000000,142.000,168.00000,254.000000,249.000000,69.000000,33.000000,83.000000,135.000000,354.000000,169.000000
5,6,329.000000,194.000000,141.000,185.00000,283.000000,185.000000,80.000000,46.000000,80.000000,121.000000,326.000000,183.000000
6,7,343.000000,275.000000,180.000,192.00000,266.000000,237.000000,108.000000,53.000000,43.000000,135.000000,360.000000,214.000000
7,8,360.000000,199.000000,177.000,166.00000,206.000000,257.000000,86.000000,34.000000,103.000000,159.000000,350.000000,227.000000
8,9,368.000000,205.000000,155.000,150.00000,152.000000,168.000000,82.000000,49.000000,147.000000,159.000000,337.000000,196.000000
9,10,310.000000,282.000000,190.000,164.09375,148.000000,164.000000,123.000000,48.000000,104.000000,148.000000,329.000000,265.000000
